# 02 — Setup del entorno (Mac · MPS)

> **Objetivo.** Verificar que el entorno tiene PyTorch con backend MPS, todas las dependencias instaladas y un smoke test que confirme que se puede entrenar.

Este notebook **no descarga modelos pesados** — solo valida el entorno.


## 1. Crear entorno virtual (fuera del notebook)

```bash
cd /Users/raul.alzate/Documents/maestria/maestria/pnl/fine-tunning
python3 -m venv .venv
source .venv/bin/activate
pip install --upgrade pip
pip install -r requirements.txt
python -m ipykernel install --user --name venv-gec --display-name "Python (venv-gec)"
```

Luego en Jupyter selecciona el kernel **Python (venv-gec)**.

---

## 2. Verificar versiones de paquetes

In [ ]:
import sys, platform
import torch
import transformers, datasets, evaluate, accelerate

print(f"Python      : {sys.version.split()[0]}")
print(f"Plataforma  : {platform.platform()}")
print(f"Procesador  : {platform.processor() or platform.machine()}")
print(f"PyTorch     : {torch.__version__}")
print(f"transformers: {transformers.__version__}")
print(f"datasets    : {datasets.__version__}")
print(f"evaluate    : {evaluate.__version__}")
print(f"accelerate  : {accelerate.__version__}")

## 3. Verificar disponibilidad de MPS

`torch.backends.mps.is_available()` debe devolver `True` en Apple Silicon (M1/M2/M3) con macOS 12.3+ y PyTorch ≥ 1.12.

`is_built()` confirma que la build de PyTorch fue compilada con soporte MPS (debería ser `True` por defecto en builds oficiales).

In [ ]:
mps_available = torch.backends.mps.is_available()
mps_built     = torch.backends.mps.is_built()

print(f"MPS available: {mps_available}")
print(f"MPS built    : {mps_built}")

if mps_available:
    device = torch.device("mps")
    print(f"\n→ Usaremos device: {device}")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"\n→ MPS no disponible, fallback CUDA: {device}")
else:
    device = torch.device("cpu")
    print(f"\n⚠ MPS y CUDA no disponibles — fallback CPU. Entrenamiento será lento.")

## 4. Habilitar fallback de operadores no soportados

Algunas ops de PyTorch aún no tienen kernel MPS. La env var `PYTORCH_ENABLE_MPS_FALLBACK=1` hace que esas ops caigan a CPU silenciosamente en vez de fallar.

> Nota: setear env vars desde Python solo afecta procesos hijos. En la práctica conviene exportarla en la terminal antes de lanzar Jupyter, pero para esta PoC el fallback rara vez se activa con T5.

In [ ]:
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
print("PYTORCH_ENABLE_MPS_FALLBACK =", os.environ.get("PYTORCH_ENABLE_MPS_FALLBACK"))

## 5. Smoke test — multiplicación matricial en MPS

Verificamos que las operaciones de tensor corren en GPU integrada.

In [ ]:
import time

shape = (4096, 4096)
x = torch.randn(shape, device=device)
y = torch.randn(shape, device=device)

# Warm-up
_ = x @ y
if device.type == "mps":
    torch.mps.synchronize()

t0 = time.time()
z = x @ y
if device.type == "mps":
    torch.mps.synchronize()
elapsed = time.time() - t0

print(f"Multiplicación {shape} @ {shape} en {device}: {elapsed*1000:.1f} ms")
print(f"Norma resultado: {z.norm().item():.2f}")

## 6. Smoke test — carga y forward pass de T5-small

Descargamos `t5-small` (~240 MB) y hacemos un forward pass dummy. Esto verifica:

1. Conectividad con Hugging Face Hub.
2. Compatibilidad de tokenizer + modelo + dispositivo.
3. Que la generación funciona end-to-end.

In [ ]:
from transformers import AutoTokenizer, T5ForConditionalGeneration

MODEL_NAME = "google-t5/t5-small"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME).to(device)
model.eval()

total_params = sum(p.numel() for p in model.parameters())
print(f"Modelo: {MODEL_NAME}")
print(f"Parámetros: {total_params/1e6:.1f}M")
print(f"Dispositivo del modelo: {next(model.parameters()).device}")

In [ ]:
# Forward pass de prueba — sin fine-tunear, t5-small base no sabe de GEC,
# pero el smoke test valida que la pipeline corre.
text = "grammar: He go to store yesterday."
inputs = tokenizer(text, return_tensors="pt").to(device)

with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=32)

print("INPUT :", text)
print("OUTPUT:", tokenizer.decode(out[0], skip_special_tokens=True))
print("\n(la salida sin fine-tune no será correcta — solo validamos pipeline)")

## 7. Crear carpeta de artefactos

In [ ]:
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ARTIFACTS = ROOT / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)
print("Artefactos en:", ARTIFACTS)

## 8. Checklist de salida

- [ ] `mps_available == True`
- [ ] Modelo `t5-small` cargado en MPS
- [ ] Forward pass devuelve string
- [ ] Carpeta `artifacts/` existe

Si todo OK → **siguiente: `03_dataset_eda.ipynb`**.
